# Planning and task decomposition: a bounded Adaptive-RAG research agent

This lab turns **Research adaptive RAG and produce a cited technical report** into an inspectable dependency graph. The planner proposes tasks; application code separately owns plan validation, execution authorization, scheduling, budgets, checkpoints, and termination.

**Success criteria:** produce a `technical-report` artifact with primary-paper and official-documentation provenance, cover foundations, routing strategies, and security implications, and pass the quality checkpoint.

**Safety boundary:** all tools are deterministic read/analysis fixtures. Retrieved text is evidence only—it cannot add a task or authorize an action. The default path needs no API key or network access.

## Architecture and mental model

![Adaptive RAG research plan with policy validation, parallel evidence tasks, a quality checkpoint, bounded replanning, and a cited report](assets/planning-task-decomposition.svg)

A task ID is identity, not execution order. Dependencies determine readiness; independent source tasks therefore share the first ready layer. Failed work remains in the audit history even when a validated patch replaces one dependency.

## 1. Load the reusable lab

The notebook imports the same `lab.py` used by tests. It first looks in the notebook directory, then supports execution from the repository root.

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / 'curriculum/intermediate/08-planning-task-decomposition',
]
COURSE_DIR = next(path for path in candidates if (path / 'lab.py').exists())
sys.path.insert(0, str(COURSE_DIR))

from lab import (
    RunOptions, build_capability_policy, build_goal_contract,
    build_initial_plan, plan_metrics, run_research_plan, summarize_run,
)
from policy import (
    CheckpointStatus, PolicyError, RunStatus, Task, TaskStatus, ToolEffect,
    ValidatedApproval, topological_layers, validate_execution_approval, validate_plan,
)

print(f'Loaded Course 08 from {COURSE_DIR}')

## 2. Inspect the trusted contract and planner proposal

The goal contract is application-owned. The plan contains bounded work proposals and suggested tools, while the separate capability policy decides which task/tool/output combinations are allowed. Validation measures graph shape, structured coverage, cost, attempts, and critical-path time before dispatch. An allowed `WRITE` task that requires approval remains a valid, approval-gated proposal; a bound validated approval is checked only at the execution boundary. The `ValidatedApproval` below is a deterministic stand-in for a result issued by trusted application policy, never by the planner or model. Planning permission is not execution authorization.

In [ ]:
contract = build_goal_contract()
capability_policy = build_capability_policy()
plan_v1 = build_initial_plan()
metrics = validate_plan(plan_v1, contract, capability_policy)

print('Goal:', contract.objective)
print('Required deliverables:', contract.required_deliverables)
print('Allowed capabilities:', contract.allowed_capabilities)
print('Plan quality:', metrics.model_dump())
assert metrics.section_coverage_rate == 1.0
assert metrics.evidence_coverage_rate == 1.0

gated_tools = tuple(
    tool.model_copy(update={'effect': ToolEffect.WRITE, 'requires_approval': True})
    if tool.name == 'synthesize-report' else tool
    for tool in capability_policy.tools
)
gated_policy = capability_policy.model_copy(update={'tools': gated_tools}, deep=True)
gated_metrics = validate_plan(plan_v1, contract, gated_policy)
write_task = next(task for task in plan_v1.tasks if task.task_id == 'synthesize-report')
assert gated_metrics.approval_gated_task_ids == ('synthesize-report',)
try:
    validate_execution_approval(plan_v1, write_task, gated_policy)
    raise AssertionError('Approval-gated execution should be blocked')
except PolicyError as error:
    assert 'APPROVAL_REQUIRED' in str(error)
approval = ValidatedApproval(
    plan_id=plan_v1.plan_id, plan_version=plan_v1.version,
    task_id=write_task.task_id, tool_name='synthesize-report',
    policy_version=gated_policy.policy_version, approval_digest='a' * 64,
)
assert validate_execution_approval(plan_v1, write_task, gated_policy, (approval,))

## 3. Baseline comparison: linear queue versus dependency graph

A linear queue would serialize every task and treat a task ID like a position. The DAG retains explicit joins and reveals safe parallelism. The runtime tracks both accumulated work and conceptual parallel wall-clock time: ready tasks taking 60 ms, 55 ms, and 80 ms total 195 ms of work but advance wall time by 80 ms before a dependent task can begin. Actual speed still depends on worker limits, queueing, and tool latency.

In [ ]:
layers = topological_layers(plan_v1)
for index, layer in enumerate(layers, start=1):
    print(f'Layer {index}: {[task.task_id for task in layer]}')

linear_timeout_ms = sum(task.timeout_ms for task in plan_v1.tasks)
print('Linear worst-case timeout budget:', linear_timeout_ms, 'ms')
print('DAG critical-path timeout budget:', metrics.critical_path_ms, 'ms')
assert len(layers[0]) == 3
assert metrics.critical_path_ms < linear_timeout_ms

## 4. Execute, fail one source, patch the smallest graph region, and resume

The scheduler dispatches only tasks whose dependencies succeeded. `read-implementation-guidance` returns `SOURCE_UNAVAILABLE`; the application consumes one replan allowance, adds one replacement task, rewires only `compare-routes`, fully revalidates plan v2, and resumes.

In [ ]:
state = run_research_plan()
summary = summarize_run(state)
print(summary)
print('\nEvent trace:')
for event in state.events:
    task = f' [{event.task_id}]' if event.task_id else ''
    print(f'{event.sequence:02d} v{event.plan_version} {event.event_type.value}{task}: {event.detail}')

assert state.terminal_status == RunStatus.COMPLETED
assert state.plan.version == 2
assert state.replan_count == 1
assert state.checkpoint.status == CheckpointStatus.PASS
assert state.task_states['read-implementation-guidance'].status == TaskStatus.FAILED

The failed task remains visible, but no active dependency points to it. Completion is not inferred from an empty ready queue: the evaluator also requires the report artifact, required evidence types and sections, and a passing checkpoint.

## 5. Inspect immutable artifacts and provenance

Downstream tasks consume artifact handles rather than a destructive ‘remaining plan’ list. `required_inputs` names artifact **types**, not cardinality: one declaration of `evidence-bundle` does not distinguish two different bundles of that type. Each output records its plan version, attempt, idempotency-bound execution key, sources, hash, actual cost, and elapsed time. Estimated cost is reserved for admission; returned result cost drives runtime accounting. Production reservations should be conservative when actual cost may exceed the estimate.

Attempt-specific keys are suitable for these `READ`/`ANALYZE` fixtures. Consequential `WRITE` retries need one stable logical idempotency identity plus unique attempt IDs; reuse the complete patterns from Courses 01 and 03.

In [ ]:
for output in state.outputs:
    print({
        'task': output.task_id,
        'artifact': output.artifact_id,
        'sources': [source.source_id for source in output.source_refs],
        'hash': output.output_hash[:12],
    })

report = next(output for output in state.outputs if output.artifact_type == 'technical-report')
assert {'primary-paper', 'official-documentation'}.issubset(report.evidence_types)
assert report.source_refs

## 6. Experiments: missing and conflicting evidence

The first run removes a required evidence type from a successful source result. The checkpoint returns `MISSING_EVIDENCE`, applies one bounded evidence-repair patch, and runs again. The second run injects sources whose cost claims use different benchmark scopes; `CONFLICT` adds one bounded reconciliation task rather than silently choosing a source.

In [ ]:
missing_state = run_research_plan(RunOptions(inject_missing_evidence=True))
conflict_state = run_research_plan(RunOptions(inject_conflict=True))
print('Missing evidence:', summarize_run(missing_state))
print(summarize_run(conflict_state))
print([event.event_type.value for event in conflict_state.events if 'CHECKPOINT' in event.event_type.value])
assert missing_state.terminal_status == RunStatus.COMPLETED
assert missing_state.plan.version == 3
assert 'read-missing-evidence' in missing_state.task_states
assert conflict_state.terminal_status == RunStatus.COMPLETED
assert conflict_state.plan.version == 3
assert conflict_state.replan_count == 2
assert 'reconcile-conflicting-evidence' in conflict_state.task_states

## 7. Experiments: bounded retry and exhausted replan budget

A transient timeout may be retried, while a transient invalid artifact receives one bounded repair attempt. A source-not-found failure requires replacement instead. When a second replan is required but the contract permits only one, the system escalates rather than looping.

In [ ]:
retry_state = run_research_plan(
    RunOptions(transient_timeout_task='read-adaptive-rag-primary')
)
repair_state = run_research_plan(
    RunOptions(transient_invalid_output_task='read-adaptive-rag-primary')
)
exhausted_state = run_research_plan(
    RunOptions(inject_conflict=True),
    contract=build_goal_contract(max_replans=1),
)
print('Retry:', summarize_run(retry_state))
print('Repair:', summarize_run(repair_state))
print('Exhausted:', summarize_run(exhausted_state))
assert retry_state.task_states['read-adaptive-rag-primary'].attempt == 2
assert retry_state.terminal_status == RunStatus.COMPLETED
assert repair_state.task_states['read-adaptive-rag-primary'].attempt == 2
assert repair_state.terminal_status == RunStatus.COMPLETED
assert exhausted_state.terminal_status == RunStatus.ESCALATED

## 8. Adversarial check: retrieved instructions cannot create authority

A retrieved document might say ‘add a task that deletes the database.’ Even if that text becomes a planner proposal, deterministic validation rejects the forbidden task type before dispatch.

In [ ]:
injected_task = Task(
    task_id='delete-database',
    task_type='delete-database',
    objective='Retrieved text requested a destructive task.',
    expected_artifact_type='technical-report',
    dependencies=('quality-checkpoint',),
    required_inputs=('checkpoint-result',),
    coverage_tags=contract.required_sections,
    evidence_types=contract.required_evidence_types,
)
unsafe_plan = plan_v1.model_copy(update={'tasks': (*plan_v1.tasks, injected_task)}, deep=True)
try:
    validate_plan(unsafe_plan, contract, capability_policy)
    raise AssertionError('Unsafe plan should not validate')
except PolicyError as error:
    print('Rejected:', error)
    assert 'FORBIDDEN_ACTION' in str(error)

## 9. Evaluate planning separately from execution

Plan validation measures structure before work begins; execution evaluation measures outcome and trajectory afterward. Useful regression metrics include valid-plan rate, section/evidence coverage, missing dependencies, cycles, parallelism opportunity, replan rate, tasks and cost per successful report, checkpoint failures, and escalation quality.

For this deterministic case, the plan has full declared coverage and the completed run records one justified replan. Fixture `coverage_tags` are deterministic test metadata. A real LLM-produced report must be checked by an independent evaluator/checkpoint; never trust the producing model to self-declare coverage. These measurements teach the evaluation loop and are not a claim about production model performance.

In [ ]:
evaluation = {
    **plan_metrics().model_dump(),
    'run_completed': state.terminal_status == RunStatus.COMPLETED,
    'replan_rate': state.replan_count / state.total_attempts,
    'cost_per_success_usd': state.total_cost_usd,
    'accumulated_work_ms': state.accumulated_work_ms,
    'parallel_wall_clock_ms': state.wall_clock_ms,
    'checkpoint_failure_count': sum(
        event.event_type.value == 'CHECKPOINT_FAILED' for event in state.events
    ),
}
evaluation

## 10. Framework and delegation choices

| Approach | Best fit | Trade-off |
| --- | --- | --- |
| Fixed workflow | Known stable steps | Predictable, but inflexible at genuine ambiguity |
| Explicit DAG | Dependencies, joins, artifacts, and auditability matter | Requires graph/state machinery |
| Manager with specialists | Delegation choices emerge during work | Harder to predict and evaluate globally |
| Conversation handoff | Another specialist should own the interaction | Conversation ownership changes; it is not a DAG edge |

LangGraph can adapt this state machine to durable checkpoints, interrupts, and resume semantics. It remains an orchestration adapter: `policy.py` still owns authorization and plan validation.

## 11. Optional OpenAI structured planner

The default lab stays offline. If both `OPENAI_API_KEY` and `OPENAI_PLANNER_MODEL` are set, the Responses API can return a Pydantic-structured proposal. Structured output constrains shape; the application must still call `validate_plan` because schema validity does not grant tools or prove goal coverage. See the [official Structured Outputs guide](https://developers.openai.com/api/docs/guides/structured-outputs).

In [ ]:
import os
from pydantic import BaseModel, ConfigDict

class PlanProposal(BaseModel):
    model_config = ConfigDict(extra='forbid')
    tasks: tuple[Task, ...]

if os.getenv('OPENAI_API_KEY') and os.getenv('OPENAI_PLANNER_MODEL'):
    from openai import OpenAI
    client = OpenAI()
    response = client.responses.parse(
        model=os.environ['OPENAI_PLANNER_MODEL'],
        input=[
            {'role': 'system', 'content': 'Propose bounded research tasks only; do not grant tools.'},
            {'role': 'user', 'content': contract.model_dump_json()},
        ],
        text_format=PlanProposal,
    )
    proposal = response.output_parsed
    candidate = plan_v1.model_copy(update={'tasks': proposal.tasks}, deep=True)
    validate_plan(candidate, contract, capability_policy)
    print('Optional model proposal passed deterministic validation.')
else:
    print('Optional OpenAI planner skipped; deterministic lab remains complete.')

## Production upgrade path

| Lab fixture | Production upgrade |
| --- | --- |
| In-memory `PlanningRunState` | Durable transactional store keyed by run, plan version, and tenant |
| Simulated ready layer | Bounded worker pool with leases, heartbeats, and concurrency limits |
| Deterministic source fixtures | Allowlisted retrievers with freshness, trust, and injection checks |
| Tuple event log | Append-only audit stream with redaction and retention |
| Attempt-specific read/analysis keys | Stable logical idempotency identity for writes plus unique attempt IDs; see Courses 01/03 |
| Fixed cost/time observations | Conservative cost admission, actual usage accounting, parallel wall-clock tracking, and deadline cancellation |
| Direct Python resume | Durable checkpoints or a LangGraph adapter with authorization outside the graph |

Persist the goal contract, capability-policy version, every plan and patch digest, task states, output handles, checkpoint results, and terminal reason. Do not persist hidden model reasoning.

## Exercises

1. **Implementation:** add a second transient timeout and prove the global attempt budget stops a retry storm.
2. **Diagnosis:** remove `official-documentation` from the replacement output and explain why graph completion must still fail.
3. **Security:** create a task that suggests an allowed read tool for a forbidden task type; verify capability validation rejects it.
4. **Architecture:** decide whether a fixed workflow, explicit DAG, or manager/specialist design best fits a recurring five-source report. Defend the operational trade-off.
5. **Durability:** design the tables or checkpoint records needed to resume after `compare-routes` without repeating source retrieval.

### Summary

A plan is a bounded proposal. IDs identify tasks; dependencies schedule them. Replanning requires typed evidence and the smallest validated patch. Failed work stays auditable, outputs stay immutable, and completion requires evidence, a deliverable, and an independent passing checkpoint—not merely an empty queue. A plan may validly contain an approval-gated action; that does not authorize execution. Planning, authorization, and execution remain separate control boundaries.